# Demo day: your six minutes, in one notebook

Run this top to bottom on Friday, one cell per beat. Every cell is a command from
`docs/DEFENCE.md`, and every one ends in a **receipt** or a **refusal**.

| Clock | Beat | Cell |
|---|---|---|
| 0:00 | Your README: one sentence and the explorer link | 1 |
| 0:45 | Gecko lists your store | 2 |
| 1:30 | The live buy | 3 |
| 2:30 | The landing: the receipt, read from the ledger | 4 |
| 3:15 | **The card**: the judge draws one | 5 |
| 4:30 | Tests and the five cases | 6 |
| 5:15 | The ADR | 7 |
| finalists | Mainnet: `show`, one espresso, `show` | 8 |
| any time | The way back: the recorded lane | 9 |

Open it with  from your repository.
Run it once on Thursday against the clock, and keep the outputs: a notebook that
already ran is your fallback if the network fails on stage. Say so if you use it.

In [ ]:
# Setup. Works from anywhere inside your my-gecko-buyer repository.
import json
import os
import subprocess
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
STORE = json.loads((ROOT / "store" / "store.json").read_text())["store"]


def run(command, tail=30, env=None):
    """Run one command from the repository root and show what it printed."""
    print(f"$ {command}")
    clean = {k: v for k, v in os.environ.items() if k != "VIRTUAL_ENV"}  # uv's own env
    done = subprocess.run(
        command,
        shell=True,
        cwd=ROOT,
        text=True,
        capture_output=True,
        env={**clean, "NO_COLOR": "1", **(env or {})},
    )
    lines = (done.stdout + done.stderr).rstrip().splitlines()
    print("\n".join(lines if len(lines) <= tail else ["..."] + lines[-tail:]))
    return done


def newest(folder, pattern):
    files = sorted((ROOT / folder).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


print("repository:", ROOT.name, "| store:", STORE)

## 1 · 0:00 · Your README

One sentence, then the explorer link to your landed devnet purchase.

In [ ]:
print("\n".join((ROOT / "README.md").read_text().splitlines()[:8]))

## 2 · 0:45 · Gecko lists your store

On stage, show it from your assistant with Gecko connected. This cell is the same call,
`list_stores`, through Gecko's keyless hosted MCP.

In [ ]:
from buyer.mcp_client import HostedGecko

answer = HostedGecko().call("list_stores", {"store": STORE, "network": "devnet"})
for store in answer.get("stores", []):
    if store.get("store") == STORE:
        print(f"{store['store']} at {store['address']}")
        print(f"  total_purchases {store.get('total_purchases')}")
        for product in store["products"]:
            print(f"  {product['name']:<12} {product['price_raw']:>9} raw")
            print(f"    mint {product['mint'][:8]}..")

## 3 · 1:30 · The live buy

Pin, prepare, seven checks, sign, verify, submit. Read the ticks out loud.

In [ ]:
buy = run('uv run buyer "one espresso" --devnet')

## 4 · 2:30 · The landing

The receipt the buyer wrote: two ledger reads, the deltas, `total_purchases` from n to n+1,
and the explorer link. Open the link.

In [ ]:
receipt = newest("receipts", "*.md")
print(receipt.relative_to(ROOT) if receipt else "no receipt yet")
print(receipt.read_text() if receipt else "")

## 5 · 3:15 · The card

The judge draws one face down and reads it out. Set `CARD`, run the cell, and narrate.
Every card runs on **devnet**, even for finalists. Stale takes about 40 seconds: say what
it is waiting for.

In [ ]:
CARD = "quantity"  # <- the judge's card: quantity, budget, tampered or stale

menu = HostedGecko().call("list_stores", {"store": STORE, "network": "devnet"})
espresso = next(
    p
    for s in menu["stores"]
    if s["store"] == STORE
    for p in s["products"]
    if p["name"].lower() == "espresso"
)
COMMANDS = {
    "quantity": 'uv run buyer "two espressos" --devnet',
    "budget": f'uv run buyer "one espresso" --budget-raw {espresso["price_raw"] // 2} --devnet',
    "tampered": 'uv run buyer "one espresso" --devnet --card tampered',
    "stale": 'uv run buyer "one espresso" --devnet --card stale',
}
card = run(COMMANDS[CARD])
refusal = newest("refusals", "*.json")
if refusal:
    print("\n", refusal.relative_to(ROOT))
    print(refusal.read_text())

### Thursday: rehearse all four cards

Not on stage: the night before. Set `REHEARSE = True` and run it once; you should see four
refusals and nothing signed. Stale waits about 40 seconds.

In [ ]:
REHEARSE = False  # <- True on Thursday

if REHEARSE:
    for name, command in COMMANDS.items():
        print(f"--- {name}")
        out = run(command, tail=3)
else:
    print("rehearsal skipped: set REHEARSE = True the night before.")

## 6 · 4:30 · Tests and the five cases

The tests trigger every refusal offline. Then the five cases and the trap, on recorded
answers. Point at one test that was red first.

In [ ]:
run("uv run pytest -q -o addopts='' --color=no", tail=1)
run("uv run buyer --cases --recorded", tail=1)
run("uv run buyer --cards --recorded", tail=1);

## 7 · 5:15 · The ADR

The decision, and what would reverse it.

In [ ]:
print(
    "\n".join(
        (ROOT / "docs" / "adr" / "0001-refusals-before-signing.md").read_text().splitlines()[:30]
    )
)

## 8 · Finalists only: mainnet

Your own wallet, registered and funded with three espressos. `show`, the buy, `show`
again: the difference is the price, exactly. **Real money.** Leave `RUN_MAINNET = False`
unless you are a finalist with a funded wallet.

In [ ]:
RUN_MAINNET = False  # <- finalists: True, on stage

if RUN_MAINNET:
    run("uv run python scripts/mainnet_wallet.py show")
    run('uv run buyer "one espresso" --mainnet --store geckocoffee')
    run("uv run python scripts/mainnet_wallet.py show")
else:
    print("mainnet skipped: RUN_MAINNET is False. Everyone else presents on devnet.")

## 9 · The way back: the recorded lane

If devnet or Gecko is down at 1:30, run this and say so: same code path, real answers we
recorded. Presenting a replay as live is the one thing that fails.

In [ ]:
run('uv run buyer "one espresso" --devnet', env={"GECKO_SOURCE": "recorded"}, tail=18);